## A full check on the statewide pipeline

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Check load_results and load_geography

In [5]:
from src.load_results import load_results

In [6]:
RESULTS_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "results"
    / "nc_primary_election_results_pct_20260303_RAW.csv"
)

In [7]:
results = load_results(RESULTS_FILE)

In [8]:
results.shape

(103517, 15)

In [9]:
results.columns.tolist()

['County',
 'Election Date',
 'Precinct',
 'Contest Group ID',
 'Contest Type',
 'Contest Name',
 'Choice',
 'Choice Party',
 'Vote For',
 'Election Day',
 'Early Voting',
 'Absentee by Mail',
 'Provisional',
 'Total Votes',
 'Real Precinct']

In [10]:
results.dtypes

County                str
Election Date         str
Precinct              str
Contest Group ID    int64
Contest Type          str
Contest Name          str
Choice                str
Choice Party          str
Vote For            int64
Election Day        int64
Early Voting        int64
Absentee by Mail    int64
Provisional         int64
Total Votes         int64
Real Precinct         str
dtype: object

## Check load_geography

In [11]:
from src.load_geography import load_geography

In [12]:
SHAPEFILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "geography"
    / "SBE_PRECINCTS_20251212"
    / "SBE_PRECINCTS_20251212.shp"
)

In [13]:
precincts = load_geography(SHAPEFILE)

In [14]:
precincts.shape

(2632, 9)

In [15]:
precincts.columns.tolist()

['id',
 'county_id',
 'prec_id',
 'enr_desc',
 'county_nam',
 'Shape_Leng',
 'Shape_Area',
 'of_prec_id',
 'geometry']

In [16]:
precincts.dtypes

id               int64
county_id        int64
prec_id            str
enr_desc           str
county_nam         str
Shape_Leng     float64
Shape_Area     float64
of_prec_id         str
geometry      geometry
dtype: object

In [17]:
precincts[precincts["county_nam"] == "BUNCOMBE"]["prec_id"]

195     01.1
254     02.1
297     03.1
383     04.1
406     05.1
        ... 
1292    68.1
1293    69.1
1295    70.1
1296     681
1297    71.1
Name: prec_id, Length: 80, dtype: str

## Check clean_results

In [18]:
from src.clean_results import clean_results

In [19]:
results = load_results(RESULTS_FILE)

results = clean_results(results)

In [20]:
results.columns.tolist()

['county',
 'election_date',
 'precinct',
 'contest_group_id',
 'contest_type',
 'contest_name',
 'choice',
 'choice_party',
 'vote_for',
 'election_day',
 'early_voting',
 'absentee_by_mail',
 'provisional',
 'total_votes',
 'real_precinct']

In [21]:
results.dtypes

county                      string
election_date       datetime64[us]
precinct                    string
contest_group_id             int64
contest_type                string
contest_name                string
choice                      string
choice_party                string
vote_for                     int64
election_day                 int64
early_voting                 int64
absentee_by_mail             int64
provisional                  int64
total_votes                  int64
real_precinct               string
dtype: object

In [22]:
results["real_precinct"].value_counts(dropna=False)

real_precinct
Y    94416
N     9101
Name: count, dtype: Int64

In [23]:
from src.validation import (
    assert_columns_exist,
    assert_unique,
    assert_no_missing,
    assert_not_empty,
)

## Check filter_contest

In [24]:
from src.filter_contest import filter_contest

In [25]:
CONTEST_NAME = "US SENATE (REP)"

# Will return real_precincts_only=True
contest = filter_contest(
    results,
    CONTEST_NAME,
)

In [26]:
results["contest_name"].drop_duplicates().sort_values().tolist()

['ALAMANCE COUNTY BOARD OF COMMISSIONERS (DEM)',
 'ALAMANCE COUNTY BOARD OF COMMISSIONERS (REP)',
 'ALAMANCE COUNTY CLERK OF SUPERIOR COURT (REP)',
 'ALAMANCE COUNTY SHERIFF (REP)',
 'ALEXANDER COUNTY BOARD OF COMMISSIONERS (REP)',
 'ALEXANDER COUNTY BOARD OF EDUCATION DISTRICT 02 (REP)',
 'ALEXANDER COUNTY CLERK OF SUPERIOR COURT (REP)',
 'ALEXANDER COUNTY REGISTER OF DEEDS (REP)',
 'ALLEGHANY COUNTY BOARD OF COMMISSIONERS (REP)',
 'ANSON COUNTY BOARD OF COMMISSIONERS DISTRICT 02 (REP)',
 'ANSON COUNTY BOARD OF EDUCATION AT-LARGE (DEM)',
 'ANSON COUNTY SHERIFF (DEM)',
 'ASHE COUNTY BOARD OF COMMISSIONERS (REP)',
 'ASHE COUNTY BOARD OF EDUCATION (REP)',
 'ASHE COUNTY SHERIFF (REP)',
 'AVERY COUNTY BOARD OF COMMISSIONERS (REP)',
 'AVERY COUNTY BOARD OF EDUCATION',
 'AVERY COUNTY CLERK OF SUPERIOR COURT (REP)',
 'BEAUFORT COUNTY BOARD OF COMMISSIONERS (REP)',
 'BEAUFORT COUNTY BOARD OF EDUCATION DISTRICT 02 (REP)',
 'BEAUFORT COUNTY BOARD OF EDUCATION DISTRICT 04 (REP)',
 'BEAUFORT COUNT

In [27]:
contest.shape

(18431, 15)

In [28]:
contest["real_precinct"].value_counts(dropna=False)

real_precinct
Y    18431
Name: count, dtype: Int64

In [29]:
contest["vote_for"].value_counts(dropna=False)

vote_for
1    18431
Name: count, dtype: int64

## Check summarize_contest

In [30]:
from src.summarize_contest import (
    summarize_precinct_results,
    summarize_precinct_winners,
    build_precinct_candidate_lists,
    summarize_contest_results,
)

In [109]:
# This gives you the full candidate-level precinct table
precinct_results = summarize_precinct_results(contest)

In [166]:
# Zero vote contests will be handled in map.js
len(precinct_results[precinct_results["contest_votes"] == 0].groupby("precinct"))

16

In [111]:
# This gives you one row per precinct for the map.
precinct_summary = summarize_precinct_winners(precinct_results)

In [112]:
precinct_summary

,county,precinct,winner,winner_votes,winner_share,runner_up,runner_up_votes,runner_up_share,contest_votes,margin_of_victory,is_tie
0,ALAMANCE,03C,Michael Whatley,289,69.471154,Donald M. (Don) Brown,38,9.134615,416,60.336538,False
1,ALAMANCE,03N,Michael Whatley,248,70.056497,Donald M. (Don) Brown,34,9.604520,354,60.451977,False
2,ALAMANCE,03N2,Michael Whatley,89,68.992248,Donald M. (Don) Brown,15,11.627907,129,57.364341,False
3,ALAMANCE,03SE,Michael Whatley,252,74.336283,Donald M. (Don) Brown,27,7.964602,339,66.371681,False
4,ALAMANCE,03SM,Michael Whatley,193,62.258065,Donald M. (Don) Brown,51,16.451613,310,45.806452,False
...,...,...,...,...,...,...,...,...,...,...,...
2628,YANCEY,07 BRU,Michael Whatley,38,63.333333,Michele Morrow,6,10.000000,60,53.333333,False
2629,YANCEY,08 CRA,Michael Whatley,194,64.882943,Donald M. (Don) Brown,35,11.705686,299,53.177258,False
2630,YANCEY,09 SOU,Michael Whatley,119,62.303665,Donald M. (Don) Brown,22,11.518325,191,50.785340,False
2631,YANCEY,10 PEN,Michael Whatley,67,75.280899,Michele Morrow,8,8.988764,89,66.292135,False


In [113]:
# This gives you the all-candidate list your popup uses.
popup_results = build_precinct_candidate_lists(precinct_results)

In [132]:
popup_results[(popup_results["county"]=="DURHAM") & (popup_results["precinct"]=="12")]["results"].value_counts()

results
[{'candidate': 'Donald M. (Don) Brown', 'votes': 0, 'share': '0'}, {'candidate': 'Elizabeth A. Temple', 'votes': 0, 'share': '0'}, {'candidate': 'Margot Dupre', 'votes': 0, 'share': '0'}, {'candidate': 'Michael Whatley', 'votes': 0, 'share': '0'}, {'candidate': 'Michele Morrow', 'votes': 0, 'share': '0'}, {'candidate': 'Richard Dansie', 'votes': 0, 'share': '0'}, {'candidate': 'Thomas Johnson', 'votes': 0, 'share': '0'}]    1
Name: count, dtype: int64

In [133]:
# This gives you the overall statewide/district-wide race totals.
contest_summary = summarize_contest_results(contest)

In [134]:
contest_summary

,choice,candidate_votes,contest_votes,vote_share
0,Michael Whatley,340816,527188,64.647905
1,Donald M. (Don) Brown,82382,527188,15.626683
2,Thomas Johnson,29501,527188,5.595916
3,Michele Morrow,29371,527188,5.571257
4,Elizabeth A. Temple,20047,527188,3.802628
5,Richard Dansie,12654,527188,2.400282
6,Margot Dupre,12417,527188,2.355327


## Check precinct crosswalk

In [202]:
from src.precinct_crosswalk import (
    prepare_results_precincts,
    prepare_geography_precincts,
    crosswalk_diagnosis,
    find_ambiguous_matches,
)

In [203]:
results_precincts = prepare_results_precincts(precinct_results)

In [204]:
results_precincts

,county,results_precinct_id,normalized_id
0,ALAMANCE,03C,03C
7,ALAMANCE,03N,03N
14,ALAMANCE,03N2,03N2
21,ALAMANCE,03SE,03SE
28,ALAMANCE,03SM,03SM
...,...,...,...
18396,YANCEY,07 BRU,07 BRU
18403,YANCEY,08 CRA,08 CRA
18410,YANCEY,09 SOU,09 SOU
18417,YANCEY,10 PEN,10 PEN


In [205]:
geo_precincts = prepare_geography_precincts(
    precincts,
    county_col="county_nam",
    precinct_col="prec_id",
)

In [206]:
geo_precincts[geo_precincts["county"] == "BUNCOMBE"].head(15)

,county,geo_precinct_id,normalized_id
195,BUNCOMBE,01.1,1.1
254,BUNCOMBE,02.1,2.1
297,BUNCOMBE,03.1,3.1
383,BUNCOMBE,04.1,4.1
406,BUNCOMBE,05.1,5.1
447,BUNCOMBE,06.1,6.1
500,BUNCOMBE,07.1,7.1
543,BUNCOMBE,31.1,31.1
556,BUNCOMBE,08.2,8.2
557,BUNCOMBE,08.3,8.3


In [207]:
matches = crosswalk_diagnosis(
    results_precincts,
    geo_precincts,
)

In [208]:
matches[matches["county"] == "BUNCOMBE"]

,county,results_precinct_id,normalized_id,geo_precinct_id,exact_match
175,BUNCOMBE,1.1,1.1,01.1,False
176,BUNCOMBE,10.1,10.1,10.1,True
177,BUNCOMBE,11.1,11.1,11.1,True
178,BUNCOMBE,12.1,12.1,12.1,True
179,BUNCOMBE,13.1,13.1,13.1,True
...,...,...,...,...,...
250,BUNCOMBE,70.1,70.1,70.1,True
251,BUNCOMBE,71.1,71.1,71.1,True
252,BUNCOMBE,8.2,8.2,08.2,False
253,BUNCOMBE,8.3,8.3,08.3,False


In [209]:
ambiguous = find_ambiguous_matches(matches)

In [210]:
ambiguous

,county,results_precinct_id,match_count


In [211]:
from src.precinct_crosswalk import (
    build_proposed_crosswalk,
    apply_crosswalk,
)

In [212]:
proposed_crosswalk = build_proposed_crosswalk(matches)

In [213]:
proposed_crosswalk.head()

,county,results_precinct_id,geo_precinct_id,match_method
0,ALAMANCE,03C,03C,exact
1,ALAMANCE,03N,03N,exact
2,ALAMANCE,03N2,03N2,exact
3,ALAMANCE,03SE,03SE,exact
4,ALAMANCE,03SM,03SM,exact


In [214]:
proposed_crosswalk["match_method"].value_counts()

match_method
exact                   2223
leading_zero             398
decimal_leading_zero      11
Name: count, dtype: int64

In [215]:
mapped_check = results_precincts.merge(
    proposed_crosswalk,
    on=[
        "county",
        "results_precinct_id",
    ],
    how="left",
)

unmatched = mapped_check[
    mapped_check["geo_precinct_id"].isna()
]

In [216]:
unmatched

,county,results_precinct_id,normalized_id,geo_precinct_id,match_method
1217,HENDERSON,CV,CV,<NA>,NaN


In [217]:
import pandas as pd

In [218]:
manual_no_geometry = pd.DataFrame(
    [
        {
            "county": "HENDERSON",
            "results_precinct_id": "CV",
            "geo_precinct_id": pd.NA,
            "match_method": "no_geometry",
        }
    ]
)

crosswalk = pd.concat(
    [
        proposed_crosswalk,
        manual_no_geometry,
    ],
    ignore_index=True,
)

In [219]:
crosswalk["match_method"].value_counts()

match_method
exact                   2223
leading_zero             398
decimal_leading_zero      11
no_geometry                1
Name: count, dtype: int64

In [220]:
crosswalk

,county,results_precinct_id,geo_precinct_id,match_method
0,ALAMANCE,03C,03C,exact
1,ALAMANCE,03N,03N,exact
2,ALAMANCE,03N2,03N2,exact
3,ALAMANCE,03SE,03SE,exact
4,ALAMANCE,03SM,03SM,exact
...,...,...,...,...
2628,YANCEY,08 CRA,08 CRA,exact
2629,YANCEY,09 SOU,09 SOU,exact
2630,YANCEY,10 PEN,10 PEN,exact
2631,YANCEY,11 PRI,11 PRI,exact


In [237]:
CROSSWALK_FILE = (
    PROJECT_ROOT
    / "data"
    / "crosswalks"
    / "precinct_crosswalk_2026.csv"
)

In [238]:
crosswalk.to_csv(CROSSWALK_FILE, index=False)

In [221]:
mapped_results = apply_crosswalk(
    precinct_results,
    crosswalk,
)

In [222]:
mapped_results[(mapped_results["county"] == "MECKLENBURG") & (mapped_results["precinct"] == "78.1")]

,county,precinct,choice,candidate_votes,contest_votes,vote_share,results_precinct_id,geo_precinct_id,match_method
11522,MECKLENBURG,78.1,Michael Whatley,19,36,52.777778,78.1,078.1,decimal_leading_zero
11523,MECKLENBURG,78.1,Donald M. (Don) Brown,10,36,27.777778,78.1,078.1,decimal_leading_zero
11524,MECKLENBURG,78.1,Richard Dansie,3,36,8.333333,78.1,078.1,decimal_leading_zero
11525,MECKLENBURG,78.1,Thomas Johnson,2,36,5.555556,78.1,078.1,decimal_leading_zero
11526,MECKLENBURG,78.1,Elizabeth A. Temple,1,36,2.777778,78.1,078.1,decimal_leading_zero
11527,MECKLENBURG,78.1,Margot Dupre,1,36,2.777778,78.1,078.1,decimal_leading_zero
11528,MECKLENBURG,78.1,Michele Morrow,0,36,0.000000,78.1,078.1,decimal_leading_zero


## Manually merge in any unmatches

In [223]:
mapped_results[
    mapped_results["geo_precinct_id"].isna()
][
    [
        "county",
        "precinct",
        "match_method",
    ]
].drop_duplicates()

,county,precinct,match_method
8519,HENDERSON,CV,no_geometry


## Check load_geography (above) and join_geography

In [224]:
type(precincts)

geopandas.geodataframe.GeoDataFrame

In [225]:
precincts.crs

<Projected CRS: EPSG:2264>
Name: NAD83 / North Carolina (ftUS)
Axis Info [cartesian]:
- X[east]: Easting (US survey foot)
- Y[north]: Northing (US survey foot)
Area of Use:
- name: United States (USA) - North Carolina - counties of Alamance; Alexander; Alleghany; Anson; Ashe; Avery; Beaufort; Bertie; Bladen; Brunswick; Buncombe; Burke; Cabarrus; Caldwell; Camden; Carteret; Caswell; Catawba; Chatham; Cherokee; Chowan; Clay; Cleveland; Columbus; Craven; Cumberland; Currituck; Dare; Davidson; Davie; Duplin; Durham; Edgecombe; Forsyth; Franklin; Gaston; Gates; Graham; Granville; Greene; Guilford; Halifax; Harnett; Haywood; Henderson; Hertford; Hoke; Hyde; Iredell; Jackson; Johnston; Jones; Lee; Lenoir; Lincoln; Macon; Madison; Martin; McDowell; Mecklenburg; Mitchell; Montgomery; Moore; Nash; New Hanover; Northampton; Onslow; Orange; Pamlico; Pasquotank; Pender; Perquimans; Person; Pitt; Polk; Randolph; Richmond; Robeson; Rockingham; Rowan; Rutherford; Sampson; Scotland; Stanly; Stokes; Sur

In [226]:
# Before merging in all precincts geography, first combine the one-row-per-precinct summary with the popup results list

map_results = precinct_summary.merge(
    popup_results,
    on=["county", "precinct"],
    how="left",
    validate="one_to_one",
)

In [227]:
map_results

,county,precinct,winner,winner_votes,winner_share,runner_up,runner_up_votes,runner_up_share,contest_votes,margin_of_victory,is_tie,results
0,ALAMANCE,03C,Michael Whatley,289,69.471154,Donald M. (Don) Brown,38,9.134615,416,60.336538,False,"[{'candidate': 'Michael Whatley', 'votes': 289..."
1,ALAMANCE,03N,Michael Whatley,248,70.056497,Donald M. (Don) Brown,34,9.604520,354,60.451977,False,"[{'candidate': 'Michael Whatley', 'votes': 248..."
2,ALAMANCE,03N2,Michael Whatley,89,68.992248,Donald M. (Don) Brown,15,11.627907,129,57.364341,False,"[{'candidate': 'Michael Whatley', 'votes': 89,..."
3,ALAMANCE,03SE,Michael Whatley,252,74.336283,Donald M. (Don) Brown,27,7.964602,339,66.371681,False,"[{'candidate': 'Michael Whatley', 'votes': 252..."
4,ALAMANCE,03SM,Michael Whatley,193,62.258065,Donald M. (Don) Brown,51,16.451613,310,45.806452,False,"[{'candidate': 'Michael Whatley', 'votes': 193..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2628,YANCEY,07 BRU,Michael Whatley,38,63.333333,Michele Morrow,6,10.000000,60,53.333333,False,"[{'candidate': 'Michael Whatley', 'votes': 38,..."
2629,YANCEY,08 CRA,Michael Whatley,194,64.882943,Donald M. (Don) Brown,35,11.705686,299,53.177258,False,"[{'candidate': 'Michael Whatley', 'votes': 194..."
2630,YANCEY,09 SOU,Michael Whatley,119,62.303665,Donald M. (Don) Brown,22,11.518325,191,50.785340,False,"[{'candidate': 'Michael Whatley', 'votes': 119..."
2631,YANCEY,10 PEN,Michael Whatley,67,75.280899,Michele Morrow,8,8.988764,89,66.292135,False,"[{'candidate': 'Michael Whatley', 'votes': 67,..."


In [228]:
# Then apply the crosswalk

map_results = apply_crosswalk(
    map_results,
    proposed_crosswalk,
)

In [229]:
map_results.columns.tolist()

['county',
 'precinct',
 'winner',
 'winner_votes',
 'winner_share',
 'runner_up',
 'runner_up_votes',
 'runner_up_share',
 'contest_votes',
 'margin_of_victory',
 'is_tie',
 'results',
 'results_precinct_id',
 'geo_precinct_id',
 'match_method']

## join_geography

In [230]:
from src.join_geography import join_precinct_geography

mapped_precincts = join_precinct_geography(
    precincts,
    map_results,
)

In [231]:
# Results should exclude any precincts that were unmatched in the crosswalk due to no geometry
mapped_precincts

,id,county_id,prec_id,enr_desc,county_nam,Shape_Leng,Shape_Area,of_prec_id,geometry,county_join,...,runner_up_votes,runner_up_share,contest_votes,margin_of_victory,is_tie,results,results_precinct_id,geo_precinct_id,match_method,participated
0,2024,79,LI-1,LINCOLN-1,ROCKINGHAM,223303.930812,2.292027e+09,NaN,"POLYGON ((1806523.304 956751.169, 1806534.127 ...",ROCKINGHAM,...,181,19.133192,946,36.786469,False,"[{'candidate': 'Michael Whatley', 'votes': 529...",LI-1,LI-1,exact,True
1,1667,79,DR,DRAPER,ROCKINGHAM,115430.785952,4.435576e+08,NaN,"POLYGON ((1788551.831 997096.871, 1788733.272 ...",ROCKINGHAM,...,59,16.619718,355,42.253521,False,"[{'candidate': 'Michael Whatley', 'votes': 209...",DR,DR,exact,True
2,2388,79,RC,ROCK CENTRAL,ROCKINGHAM,237535.047394,1.572654e+09,NaN,"POLYGON ((1775658.388 946627.913, 1775456.261 ...",ROCKINGHAM,...,252,20.437956,1233,32.846715,False,"[{'candidate': 'Michael Whatley', 'votes': 657...",RC,RC,exact,True
3,2649,79,WS-1,WESTERN-1,ROCKINGHAM,287741.993717,1.728622e+09,NaN,"POLYGON ((1695965.848 950230.626, 1695968.951 ...",ROCKINGHAM,...,178,18.445596,965,33.471503,False,"[{'candidate': 'Michael Whatley', 'votes': 501...",WS-1,WS-1,exact,True
4,3100,79,ST,STONEVILLE,ROCKINGHAM,234984.364396,1.810038e+09,NaN,"POLYGON ((1723435.755 980911.515, 1723368.983 ...",ROCKINGHAM,...,190,18.428710,1031,37.633366,False,"[{'candidate': 'Michael Whatley', 'votes': 578...",ST,ST,exact,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627,3145,92,05-10,05-10,WAKE,54128.846329,1.145843e+08,NaN,"POLYGON ((2037950.618 753669.957, 2037929.585 ...",WAKE,...,9,14.516129,62,43.548387,False,"[{'candidate': 'Michael Whatley', 'votes': 36,...",05-10,05-10,exact,True
2628,3146,71,PL18,Penderlea,PENDER,256833.239095,2.497973e+09,NaN,"POLYGON ((2331510.751 313349.517, 2329331.826 ...",PENDER,...,59,13.348416,442,51.131222,False,"[{'candidate': 'Michael Whatley', 'votes': 285...",PL18,PL18,exact,True
2629,3147,71,BW01,Burgaw,PENDER,452401.512499,3.478623e+09,NaN,"POLYGON ((2349354.689 262479.965, 2349376.936 ...",PENDER,...,86,10.093897,852,59.507042,False,"[{'candidate': 'Michael Whatley', 'votes': 593...",BW01,BW01,exact,True
2630,3148,71,RP21,Rocky Point,PENDER,440134.337501,4.262947e+09,NaN,"POLYGON ((2321434.642 223017.728, 2321401.381 ...",PENDER,...,99,11.942099,829,53.196622,False,"[{'candidate': 'Michael Whatley', 'votes': 540...",RP21,RP21,exact,True


In [ ]:
# For map pop up 
# Move into join_geography.py
# mapped_precincts["enr_desc"] = mapped_precincts["enr_desc"].str.title()

In [ ]:
# For map pop up 
# Move into join_geography.py
# mapped_precincts["map_key"] = mapped_precincts["county"] + "_" + mapped_precincts["precinct"]

## Export GeoJSON

In [234]:
WEB_COLUMNS = [
    "county",
    "precinct",
    "enr_desc",
    "winner",
    "winner_votes",
    "winner_share",
    "runner_up",
    "runner_up_votes",
    "runner_up_share",
    "contest_votes",
    "margin_of_victory",
    "is_tie",
    "results",
    "participated",
    "map_key",
    "geometry"
]

In [235]:
from src.export_geojson import export_geojson
import pydash


WEB_FILE = (
    PROJECT_ROOT
    / "map"
    / "data"
    / f"{pydash.snake_case(CONTEST_NAME)}.geojson"
)

web_precincts = export_geojson(
    mapped_precincts,
    WEB_FILE,
    columns=WEB_COLUMNS,
)

In [236]:
web_precincts

,county,precinct,enr_desc,winner,winner_votes,winner_share,runner_up,runner_up_votes,runner_up_share,contest_votes,margin_of_victory,is_tie,results,participated,map_key,geometry
0,ROCKINGHAM,LI-1,Lincoln-1,Michael Whatley,529,55.919662,Donald M. (Don) Brown,181,19.133192,946,36.786469,False,"[{'candidate': 'Michael Whatley', 'votes': 529...",True,ROCKINGHAM_LI-1,"POLYGON ((-79.65716 36.3769, -79.65712 36.377,..."
1,ROCKINGHAM,DR,Draper,Michael Whatley,209,58.873239,Donald M. (Don) Brown,59,16.619718,355,42.253521,False,"[{'candidate': 'Michael Whatley', 'votes': 209...",True,ROCKINGHAM_DR,"POLYGON ((-79.71919 36.48737, -79.71858 36.487..."
2,ROCKINGHAM,RC,Rock Central,Michael Whatley,657,53.284672,Donald M. (Don) Brown,252,20.437956,1233,32.846715,False,"[{'candidate': 'Michael Whatley', 'votes': 657...",True,ROCKINGHAM_RC,"POLYGON ((-79.76172 36.34849, -79.7624 36.3477..."
3,ROCKINGHAM,WS-1,Western-1,Michael Whatley,501,51.917098,Donald M. (Don) Brown,178,18.445596,965,33.471503,False,"[{'candidate': 'Michael Whatley', 'votes': 501...",True,ROCKINGHAM_WS-1,"POLYGON ((-80.03242 36.35641, -80.03241 36.356..."
4,ROCKINGHAM,ST,Stoneville,Michael Whatley,578,56.062076,Donald M. (Don) Brown,190,18.428710,1031,37.633366,False,"[{'candidate': 'Michael Whatley', 'votes': 578...",True,ROCKINGHAM_ST,"POLYGON ((-79.94013 36.44142, -79.94036 36.441..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627,WAKE,05-10,05-10,Michael Whatley,36,58.064516,Donald M. (Don) Brown,9,14.516129,62,43.548387,False,"[{'candidate': 'Michael Whatley', 'votes': 36,...",True,WAKE_05-10,"POLYGON ((-78.87199 35.82074, -78.87206 35.820..."
2628,PENDER,PL18,Penderlea,Michael Whatley,285,64.479638,Donald M. (Don) Brown,59,13.348416,442,51.131222,False,"[{'candidate': 'Michael Whatley', 'votes': 285...",True,PENDER_PL18,"POLYGON ((-77.89829 34.60592, -77.90554 34.605..."
2629,PENDER,BW01,Burgaw,Michael Whatley,593,69.600939,Donald M. (Don) Brown,86,10.093897,852,59.507042,False,"[{'candidate': 'Michael Whatley', 'votes': 593...",True,PENDER_BW01,"POLYGON ((-77.84096 34.46559, -77.84089 34.465..."
2630,PENDER,RP21,Rocky Point,Michael Whatley,540,65.138721,Donald M. (Don) Brown,99,11.942099,829,53.196622,False,"[{'candidate': 'Michael Whatley', 'votes': 540...",True,PENDER_RP21,"POLYGON ((-77.93499 34.35802, -77.9351 34.3580..."


In [ ]:
participated = web_precincts[
    web_precincts["participated"]
].copy()

west, south, east, north = participated.total_bounds

contest_bounds = [
    [west, south],
    [east, north]
]

print(contest_bounds)